# RAG from Scratch to Production — Episode 20/20
## CI/CD and deployment: would the tests have caught it?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/20_ci_cd_deployment.ipynb)

The finale. Over 19 episodes, this series found real bugs in the [production app](https://github.com/JakkulaMahendhar/rag-chatbot): a parser dropping tables, a tokenizer losing the last word of every question, hybrid weights that didn't mean what they said, conversations without an owner, and more.

The production app **has** CI: GitHub Actions runs its test suite on every push and pull request. So the question for the last episode is simple:

> **Would CI have caught any of these bugs?**

**In this episode:**
1. Check what production's CI actually tests.
2. Write a small test suite that catches this series' bugs, and run it from this notebook.
3. Split tests into **fast** (every push) and **slow** (AI calls, on a schedule).
4. A deployment checklist from all 20 episodes.

**No API key needed.**

## 1. What production's CI tests

From the production app's `.github/workflows/ci.yml` and `tests/` folder:
- ✅ CI runs on every push and pull request to `master`, with real PostgreSQL and ChromaDB services, and even tests **each commit** of a push separately.
- 23 test functions run. **9 of 23 test files are skipped** as "pre-existing failures", including `test_bm25.py`, `test_chunker.py`, `test_embedding.py` and `test_retrieval.py`.

What the running tests cover:

| Area | Tests that run | Would it catch this series' bugs? |
|---|---|---|
| Auth, login, tokens | 7 | ✅ Good coverage |
| Document ownership across users | 5 | ✅ Including cross-user access |
| Parsing | 1 (`.txt` only) | ❌ No PDF, Word table or encoding tests (Episode 3) |
| Retrieval: chunking, BM25, embeddings, hybrid | 0 (all skipped) | ❌ Tokenizer bug (Ep 7), hybrid scaling (Ep 8) |
| Prompt | 2 | ❌ They check the rule **text** exists, not what the model does (Ep 13) |
| Answers, refusals, guard | 0 | ❌ (Ep 11, 15) |
| Conversations | 0 | ❌ No ownership check (Ep 18) |

**For the bugs this series found in retrieval and answers, CI would have caught none of them.** That's not unusual: tests usually cover the parts that are easy to test (APIs, auth) and skip the parts that are hard to test (search quality). Episode 16 showed those parts **can** be tested cheaply. Let's turn that into a test suite.

## 2. A test suite for RAG bugs

### The code under test

A small module with the pieces where bugs were found. `RAG_IMPL=production` gives production's versions, `RAG_IMPL=fixed` the fixed versions from this series.

In [ ]:
%pip install -q pytest pymupdf python-docx rank-bm25 sentence-transformers

In [1]:
import os
os.makedirs("tests", exist_ok=True)                 # where the test files go
os.makedirs(".github/workflows", exist_ok=True)     # where the CI workflow goes

In [2]:
%%writefile rag_pipeline.py
"""Pieces of a RAG pipeline where this series found bugs. Set RAG_IMPL=production or fixed."""
import os, re
from pathlib import Path
import fitz
from docx import Document

IMPL = os.getenv("RAG_IMPL", "fixed")


# ── Parsing (Episode 3) ──────────────────────────────────────────────
def parse_production(path: Path) -> str:
    if path.suffix == ".pdf":
        with fitz.open(path) as document:
            return "".join(page.get_text() for page in document)
    if path.suffix == ".docx":
        return "\n".join(p.text for p in Document(path).paragraphs)
    with open(path, encoding="utf-8") as f:
        return f.read()


def parse_fixed(path: Path) -> str:
    if path.suffix == ".pdf":
        with fitz.open(path) as document:
            return "\n".join(page.get_text() for page in document)
    if path.suffix == ".docx":
        blocks = []
        for block in Document(path).iter_inner_content():
            if hasattr(block, "rows"):
                blocks += [f"{r.cells[0].text}: {r.cells[1].text}." for r in block.rows[1:]]
            elif block.text.strip():
                blocks.append(block.text)
        return "\n".join(blocks)
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace")


# ── Ingestion checks (Episodes 3 and 16) ─────────────────────────────
def ingestion_problems_production(path: Path) -> list[str]:
    return []                                   # production has no ingestion checks


def ingestion_problems_fixed(path: Path, min_chars_per_page: int = 100) -> list[str]:
    if path.suffix != ".pdf":
        return []
    with fitz.open(path) as document:
        return [f"page {n} has only {len(p.get_text().strip())} characters"
                for n, p in enumerate(document, start=1) if len(p.get_text().strip()) < min_chars_per_page]


# ── Tokenizing for BM25 (Episode 7) ──────────────────────────────────
def tokenize_production(text: str) -> list[str]:
    return text.lower().split()


def tokenize_fixed(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


# ── Conversations (Episode 18) ───────────────────────────────────────
class ConversationStoreProduction:
    def __init__(self):
        self.messages = {}

    def add(self, conversation_id, user_id, text):
        self.messages.setdefault(conversation_id, []).append(text)

    def history(self, conversation_id, user_id):
        return self.messages.get(conversation_id, [])


class ConversationStoreFixed:
    def __init__(self):
        self.messages, self.owners = {}, {}

    def _check(self, conversation_id, user_id):
        if self.owners.setdefault(conversation_id, user_id) != user_id:
            raise PermissionError("not your conversation")

    def add(self, conversation_id, user_id, text):
        self._check(conversation_id, user_id)
        self.messages.setdefault(conversation_id, []).append(text)

    def history(self, conversation_id, user_id):
        self._check(conversation_id, user_id)
        return self.messages.get(conversation_id, [])


_fixed = IMPL == "fixed"
parse = parse_fixed if _fixed else parse_production
ingestion_problems = ingestion_problems_fixed if _fixed else ingestion_problems_production
tokenize = tokenize_fixed if _fixed else tokenize_production
ConversationStore = ConversationStoreFixed if _fixed else ConversationStoreProduction

Writing rag_pipeline.py


### Test documents

A pytest **fixture** builds Episode 16's documents once per test run: a normal PDF, a pure scan and a stamped scan, a Word file with a table, and a text file saved on Windows.

In [3]:
%%writefile tests/conftest.py
import fitz, pytest
from docx import Document


def _scan(path, text, stamp=None):
    page = fitz.open().new_page()
    page.insert_textbox(fitz.Rect(72, 72, 520, 400), text, fontsize=16)
    scan = fitz.open()
    scan_page = scan.new_page()
    scan_page.insert_image(fitz.Rect(0, 0, 595, 842), stream=page.get_pixmap(dpi=100).tobytes("png"))
    if stamp:
        scan_page.insert_text((400, 820), stamp, fontsize=8)
    scan.save(path, deflate=True)


@pytest.fixture(scope="session")
def docs(tmp_path_factory):
    folder = tmp_path_factory.mktemp("docs")

    pdf = fitz.open()
    pdf.new_page().insert_textbox(fitz.Rect(72, 80, 520, 760),
        "Remote Work\nEmployees may work remotely up to 2 days per week. Remote days must be logged in "
        "the HR portal by Friday of the previous week.", fontsize=12)
    pdf.save(folder / "handbook.pdf")

    _scan(folder / "parental_leave.pdf", "Parental Leave\nNew parents receive 16 weeks of fully paid leave.")
    _scan(folder / "it_notice.pdf", "VPN error E-4031: reset your token in Okta.", stamp="Scanned 12/03/2026 - Mailroom")

    doc = Document()
    doc.add_paragraph("NovaTech pays the following monthly benefits to every employee.")
    table = doc.add_table(rows=1, cols=2)
    table.rows[0].cells[0].text, table.rows[0].cells[1].text = "Benefit", "Amount"
    for benefit, amount in [("Gym membership", "50 USD per month"), ("Phone bill", "30 USD per month")]:
        row = table.add_row().cells
        row[0].text, row[1].text = benefit, amount
    doc.save(folder / "benefits.docx")

    (folder / "cafeteria.txt").write_bytes("Lunch vouchers are worth 8 € per working day.\n".encode("cp1252"))
    return folder

Writing tests/conftest.py


### The tests

One test per bug class this series found. Each test states, in plain words, what "working" means.

In [4]:
%%writefile tests/test_ingestion.py
import pytest
from rag_pipeline import parse, ingestion_problems


def test_word_tables_are_kept(docs):
    text = parse(docs / "benefits.docx")
    assert "Gym membership" in text and "50 USD" in text


def test_windows_text_files_are_read(docs):
    assert "8 €" in parse(docs / "cafeteria.txt")


@pytest.mark.parametrize("name", ["parental_leave.pdf", "it_notice.pdf"])
def test_scanned_pages_are_flagged(docs, name):
    """A scan must never be indexed silently as if it had text."""
    assert ingestion_problems(docs / name), f"{name} was accepted without a warning"

Writing tests/test_ingestion.py


In [5]:
%%writefile tests/test_search.py
import re
import numpy as np
import pytest
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from rag_pipeline import parse, tokenize

SECTIONS = {
    "Annual Leave": "Full-time employees receive 24 days of paid annual leave. Book leave with form HR-12.",
    "Parental Leave": "New parents receive 16 weeks of fully paid leave. Submit form HR-17 in advance.",
    "Expenses": "Travel expenses are claimed with form EXP-22 within 30 days.",
    "IT Support": "If the VPN shows error E-4031, reset your access token in the Okta portal.",
}


@pytest.mark.parametrize("question, section", [
    ("Which form is HR-12?", "Annual Leave"),
    ("Which form do I need for parental leave? Is it HR-17?", "Parental Leave"),
])
def test_codes_at_the_end_of_a_question_are_found(question, section):
    """Episode 7: the last word of a question ends with '?'. It must still match."""
    bm25 = BM25Okapi([tokenize(text) for text in SECTIONS.values()])
    best = list(SECTIONS)[int(np.argmax(bm25.get_scores(tokenize(question))))]
    assert best == section


KNOWN_FACTS = [
    ("handbook.pdf", "How many days per week can I work remotely?", "2 days"),
    ("handbook.pdf", "When must remote days be logged?", "friday"),
    ("benefits.docx", "How much is the gym membership benefit?", "50 usd"),
    ("benefits.docx", "How much is paid towards the phone bill?", "30 usd"),
    ("cafeteria.txt", "How much are lunch vouchers worth?", "8 €"),
]


@pytest.fixture(scope="module")
def index(docs):
    chunks = []
    for path in docs.iterdir():
        try:
            text = parse(path)
        except Exception:
            continue                          # a crashed parser loses the document
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]
        chunks += [" ".join(sentences[i:i + 2]) for i in range(0, len(sentences), 2)]
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model, chunks, model.encode(chunks, normalize_embeddings=True)


@pytest.mark.parametrize("document, question, fact", KNOWN_FACTS)
def test_known_facts_are_retrievable(index, document, question, fact):
    """Episode 16: after parsing and chunking, search must still find each known fact."""
    model, chunks, vectors = index
    top3 = [chunks[i] for i in np.argsort(-(vectors @ model.encode(question, normalize_embeddings=True)))[:3]]
    assert any(fact in c.lower() for c in top3), f"'{fact}' from {document} is no longer findable"

Writing tests/test_search.py


In [6]:
%%writefile tests/test_access.py
import pytest
from rag_pipeline import ConversationStore


def test_conversations_belong_to_their_owner():
    """Episode 18: another user must not read a conversation, even with its id."""
    store = ConversationStore()
    store.add("conversation-1", user_id=1, text="I'm on a performance plan. When is my salary review?")
    with pytest.raises(PermissionError):
        store.history("conversation-1", user_id=2)

Writing tests/test_access.py


## 3. Run it: production-style code vs fixed code

In [7]:
import os, subprocess, sys

def run_tests(impl):
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "tests", "-q", "--tb=no", "-rf", "--color=no", "-p", "no:cacheprovider"],
        env={**os.environ, "RAG_IMPL": impl}, capture_output=True, text=True)
    lines = [l for l in result.stdout.splitlines() if l.startswith("FAILED") or " passed" in l or " failed" in l]
    print(f"══ RAG_IMPL={impl}")
    print("\n".join(lines), "\n")

run_tests("production")
run_tests("fixed")

══ RAG_IMPL=production
FAILED tests/test_access.py::test_conversations_belong_to_their_owner - Faile...
FAILED tests/test_ingestion.py::test_word_tables_are_kept - AssertionError: a...
FAILED tests/test_ingestion.py::test_windows_text_files_are_read - UnicodeDec...
FAILED tests/test_ingestion.py::test_scanned_pages_are_flagged[parental_leave.pdf]
FAILED tests/test_ingestion.py::test_scanned_pages_are_flagged[it_notice.pdf]
FAILED tests/test_search.py::test_codes_at_the_end_of_a_question_are_found[Which form is HR-12?-Annual Leave]
FAILED tests/test_search.py::test_codes_at_the_end_of_a_question_are_found[Which form do I need for parental leave? Is it HR-17?-Parental Leave]
FAILED tests/test_search.py::test_known_facts_are_retrievable[benefits.docx-How much is the gym membership benefit?-50 usd]
FAILED tests/test_search.py::test_known_facts_are_retrievable[benefits.docx-How much is paid towards the phone bill?-30 usd]
FAILED tests/test_search.py::test_known_facts_are_retrievable[cafeter

══ RAG_IMPL=fixed
12 passed, 5 warnings in 6.73s 


Against production-style code, the suite fails, and **each failure names the bug**:
- `test_word_tables_are_kept`: the Word table is dropped (Episode 3).
- `test_windows_text_files_are_read`: the `€` crash (Episode 3).
- `test_scanned_pages_are_flagged`: both scans accepted silently (Episodes 3 and 16).
- `test_codes_at_the_end_of_a_question_are_found`: the `.split()` tokenizer (Episode 7).
- `test_known_facts_are_retrievable`: the facts lost because of the above (Episode 16).
- `test_conversations_belong_to_their_owner`: the missing ownership check (Episode 18).

Against the fixed code, everything passes. **No LLM calls, a few seconds, and it runs on every push.**

## 4. Fast tests vs slow tests

Not every check belongs on every push:

| Kind | Examples | When to run | Cost |
|---|---|---|---|
| **Fast**: no AI calls | Parsing, ingestion checks, tokenizer, known-facts retrieval, ownership | Every push and pull request | Seconds, free |
| **Slow**: real AI calls | Answers and refusals (Ep 16), the guard (Ep 15), planted instructions (Ep 13), model swaps (Ep 14) | Nightly, before a release, or on demand | Minutes, uses API quota |

A GitHub Actions workflow for both:

In [8]:
%%writefile .github/workflows/rag-quality.yml
name: RAG quality

on:
  push:
  pull_request:
  schedule:
    - cron: "0 3 * * *"          # nightly
  workflow_dispatch:              # run by hand before a release

jobs:
  fast:                           # no AI calls: runs on every change
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - uses: actions/cache@v4    # don't re-download the embedding model every run
        with:
          path: ~/.cache/huggingface
          key: hf-models-v1
      - run: pip install -r requirements.txt pytest
      - run: pytest tests -q

  slow:                           # real AI calls: nightly or by hand only
    if: github.event_name == 'schedule' || github.event_name == 'workflow_dispatch'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt pytest
      - run: pytest tests_llm -q
        env:
          GEMINI_API_KEY: ${{ secrets.GEMINI_API_KEY }}   # never printed, never available to forks' pull requests

Writing .github/workflows/rag-quality.yml


Two details worth copying:
- **Cache the embedding model**, or every run re-downloads it.
- **Keep API keys in GitHub Secrets**, and run AI tests only on schedule or by hand. Secrets aren't passed to pull requests from forks, so a stranger's pull request can't spend your quota or print your key.

And one from production's own CI worth keeping: it lists **every commit** in a push and tests each one, so a broken commit in the middle of a push can't hide behind a fixed one at the end.

## 5. The deployment checklist

Everything this series measured, as questions to ask before shipping a RAG app:

**Data and ingestion**
- [ ] Do Word tables, PDFs with headers, and non-UTF-8 text survive parsing? (Ep 3)
- [ ] Are scans flagged by characters per page, not silently "completed"? (Ep 3, 16)
- [ ] Is there a known-facts test per important document? (Ep 4, 16)

**Search**
- [ ] Is the BM25 tokenizer the same for chunks and questions, and does it strip punctuation? (Ep 7)
- [ ] Are hybrid scores on the same scale before they're weighted? (Ep 8)
- [ ] Was the embedding model tested on your own abbreviations? (Ep 5)

**Answers**
- [ ] Is temperature set explicitly, for every provider? (Ep 14)
- [ ] Is the refusal sentence identical everywhere, so code can detect it? (Ep 13)
- [ ] Is the reranker threshold tested on questions it must answer **and** refuse? (Ep 11)
- [ ] Is the hallucination guard skipped on refusals? (Ep 15)

**The app**
- [ ] Do async endpoints `await` or thread every blocking call? (Ep 17)
- [ ] Is every object a user can name (documents, conversations) checked for ownership? (Ep 18)
- [ ] Is every endpoint that costs money rate-limited, per user? (Ep 18)

**Deployment**
- [ ] Is every store on persistent storage, tested with a real container recreate? (Ep 19)
- [ ] Does startup check that the database and the vector store agree? (Ep 19)
- [ ] Can the web service and the worker both reach the vector database and the uploaded files?
- [ ] Is `JWT_SECRET_KEY` long, random, and set only as a secret?

For the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s Render setup, the last group matters most: its `render.yaml` explains that the free plan has **no persistent disk** (uploads, vectors and the BM25 index are lost on redeploy while PostgreSQL keeps the records, Episode 19's pattern), and the file as written defines no ChromaDB service and no shared storage between the web service and the worker. Check those settings in your hosting dashboard before relying on a deployment.

## 6. The whole series, in one table

| # | Episode | What we measured |
|---|---|---|
| 1 | Why RAG | A plain LLM couldn't answer about the handbook. 20 lines of RAG could, until "WFH" |
| 2 | Architecture | One question = 4 LLM calls in production |
| 3 | Parsing | 4 silent parser bugs: tables, encodings, scans, headers |
| 4 | Chunking | Fixed-size 6/8 facts vs recursive 8/8. Overlap: 0 extra facts |
| 5 | Embeddings | 9/10 vs keywords 6/10, but "WFH" still failed |
| 6 | Vector DB | numpy searched 500,000 vectors in 8 ms. Forget one filter, leak a salary file |
| 7 | BM25 | One `?` took keyword search from 9/11 to 6/11 |
| 8 | Hybrid search | Combined: 9/11, worse than embeddings alone, until both scores shared a scale (11/11) |
| 9 | Query enhancement | Skipping the rewrite without history: same answers, 35% fewer calls |
| 10 | Reranking | Didn't improve ranking here, but scored off-topic questions 0.0000 |
| 11 | Threshold | One cutoff refused correct answers. Three bands: never confidently wrong |
| 12 | Context | 1 chunk: 10/10. 3 chunks: 8/10, the model froze between near-twins |
| 13 | Prompts | Planted instructions obeyed 3/6 times. A prompt isn't a security boundary |
| 14 | Swappable LLMs | No temperature set: 5 different answers in 5 runs |
| 15 | Guard | Caught 9/10 fakes, changed 0/24 real answers |
| 16 | Evaluation | One parser swap: known facts 8/10 → 4/10, faithfulness still 10/10 |
| 17 | FastAPI | One chat froze the server: `/health` waited 2.9 s |
| 18 | Auth | Documents safe. Conversations had no owner, chat had no limit |
| 19 | Docker | Database: 2 completed. Vector store: 0 chunks |
| 20 | CI/CD | Production's CI would have caught none of the retrieval and answer bugs. A few seconds of tests would have caught them all |

**The rule of the whole series: measure it.** Every confident claim (more context is safer, always rerank, hybrid gets the best of both, the guard protects you) turned out to depend on the data. Write down what "working" means for your documents, and test it on every change.

Thank you for following all 20 episodes. 🙌

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)